# 🚀 Elastic 9.x Real-World Vector Search Demo
본 데모는 실제 한국어 데이터 기반으로 최신 버전에 도입된 **인프라 비용 최적화(bfloat16, BBQ)**, **ACORN-1 희소 필터 속도 향상**, 그리고 **Retriever API / ES|QL 기반 하이브리드 검색**을 증명합니다.

In [ ]:
import os
import time
import getpass
import warnings
import urllib3
from elasticsearch import Elasticsearch
from elasticsearch.exceptions import ElasticsearchWarning

urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)
warnings.simplefilter('ignore', ElasticsearchWarning)

print("🔌 [Live Demo] Elasticsearch 클러스터 연결")
es_url = os.environ.get("ES_URL") or input("ES URL: ") or "https://localhost:9200"
es_user = os.environ.get("ES_USER") or input("Username: ") or "elastic"
es_pass = os.environ.get("ES_PASSWORD") or getpass.getpass("Password: ")

es = Elasticsearch(es_url, basic_auth=(es_user, es_pass), verify_certs=False)
es.ping()

INDEX_REAL_LEGACY = "demo-real-legacy-8x"
INDEX_REAL_MODERN = "demo-real-modern-9x"
VECTOR_DIMS = 1024
doc_count = es.count(index=INDEX_REAL_MODERN)['count']
print(f"📊 현재 적재된 문서 수: {doc_count:,} 건")

## 💡 [Part 1] 하드웨어 리소스(Storage & RAM) A/B 비교
`bfloat16` 적용으로 인한 디스크 절감과, `BBQ(Better Binary Quantization)` 1-bit 압축으로 인한 RAM 메모리의 32배 감소 효과를 실제 인덱스 통계로 확인합니다.

In [ ]:
stats_legacy = es.indices.stats(index=INDEX_REAL_LEGACY)['indices'][INDEX_REAL_LEGACY]['total']['store']['size_in_bytes']
stats_modern = es.indices.stats(index=INDEX_REAL_MODERN)['indices'][INDEX_REAL_MODERN]['total']['store']['size_in_bytes']

ram_legacy_mb = (doc_count * VECTOR_DIMS * 4) / (1024 * 1024)
ram_modern_mb = (doc_count * VECTOR_DIMS / 8) / (1024 * 1024)

print(f"💾 [디스크/스토리지 실제 사용량]")
print(f" - 과거 8.x (float32): {stats_legacy / (1024*1024):.2f} MB")
print(f" - 최신 9.x (bfloat16): {stats_modern / (1024*1024):.2f} MB  🔥 (약 {((stats_legacy - stats_modern) / stats_legacy) * 100:.1f}% 절감)\n")

print(f"🧠 [검색 속도 유지를 위한 RAM 상시 적재량]")
print(f" - 과거 8.x: {ram_legacy_mb:.2f} MB")
print(f" - 최신 9.x: {ram_modern_mb:.2f} MB 🔥 (정확히 32배 메모리 다이어트!)")

## 💡 [Part 2] 1% 희소 조건 필터링 속도 A/B 테스트 (ACORN-1)
전체 문서의 1%만 접근 가능한 권한(`access_level: premium`)으로 필터링을 걸었을 때, 과거 HNSW의 속도 저하 한계를 극복한 최신 **ACORN-1 알고리즘**의 속도를 벤치마킹합니다.

In [ ]:
query_text = "인공지능과 머신러닝 반도체 기술 동향"
restrictive_filter = {"term": {"access_level": "premium"}} 

legacy_query = {"knn": {"field": "title_vector", "query_vector_builder": {"text_embedding": {"model_id": "my-jina-embedding", "model_text": query_text}}, "k": 10, "num_candidates": 100, "filter": restrictive_filter}, "_source": False}
modern_query = {"query": {"bool": {"must": {"semantic": {"field": "content_semantic", "query": query_text}}, "filter": restrictive_filter}}, "_source": False}

# Warm-up
for _ in range(3): es.search(index=INDEX_REAL_LEGACY, body=legacy_query); es.search(index=INDEX_REAL_MODERN, body=modern_query)

ITERATIONS = 30
start = time.time(); [es.search(index=INDEX_REAL_LEGACY, body=legacy_query) for _ in range(ITERATIONS)]; latency_legacy = (time.time() - start) * 1000 / ITERATIONS
start = time.time(); [es.search(index=INDEX_REAL_MODERN, body=modern_query) for _ in range(ITERATIONS)]; latency_modern = (time.time() - start) * 1000 / ITERATIONS

print(f"🗣️ 질문: '{query_text}'")
print(f"⏳ 과거 8.x (Post-filter) 평균 응답 시간: {latency_legacy:.2f} ms")
print(f"🚀 최신 9.x (ACORN-1) 평균 응답 시간: {latency_modern:.2f} ms  🔥 ({latency_legacy/latency_modern:.1f}배 압도적 속도 향상!)")

## 💡 [Part 3] 실제 하이브리드 검색 품질 시연 (Retriever API)
복잡한 쿼리 가중치 튜닝 없이, **Retriever API** 블록 안에 키워드 검색과 시맨틱 검색을 넣으면 내부적으로 RRF가 결합하고, Jina Reranker가 최종 순위를 정밀하게 조정합니다.

In [ ]:
hybrid_query = {
    "retriever": {
        "text_similarity_reranker": {
            "retriever": {"rrf": {"retrievers": [{"standard": {"query": {"bool": {"must": {"match": {"title": query_text}}, "filter": restrictive_filter}}}}, {"standard": {"query": {"bool": {"must": {"semantic": {"field": "content_semantic", "query": query_text}}, "filter": restrictive_filter}}}}]}},
            "field": "title", "inference_id": "my-jina-reranker", "inference_text": query_text
        }
    }, "_source": ["title", "category", "access_level"]
}

res = es.search(index=INDEX_REAL_MODERN, body=hybrid_query)
for i, hit in enumerate(res['hits']['hits'][:3], 1): 
    print(f"{i}. [Score: {hit['_score']:.4f}] [{hit['_source']['category']} | {hit['_source']['access_level']}] {hit['_source']['title']}")

## 💡 [Part 4] 데이터 엔지니어를 위한 ES|QL 하이브리드
파이프라인 언어인 **ES|QL**을 사용하여 `FORK` (조건 병렬 탐색)와 `FUSE` (결과 융합)를 단 몇 줄의 직관적인 쿼리로 구현합니다.

In [ ]:
esql_query = f"""
    FROM {INDEX_REAL_MODERN}
    | METADATA _score
    | FORK
        (WHERE access_level == "premium" AND title LIKE "*인공지능*")
        (WHERE access_level == "premium" AND KNN(content_semantic, "{query_text}"))
    | FUSE
    | LIMIT 3
"""
esql_res = es.esql.query(body={"query": esql_query})

for i, row in enumerate(esql_res['values'], 1): 
    print(f"{i}. [Score: {row[-1]:.4f}] [{row[3]} | {row[4]}] {row[2]}")